# Experiment 03: XGBoost + Feature Engineering

This experiment keeps the feature engineering from Experiment 02 and changes the model to XGBoost.

The goal is to see whether a stronger tree-based model can improve the validation accuracy.


In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

TRAIN_PATH = "../data/train.csv"

train = pd.read_csv(TRAIN_PATH)

train.head()


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
def add_features(df):
    df = df.copy()

    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

    df["Title"] = df["Name"].str.extract(r",\s*([^.]*)\.", expand=False).str.strip()

    df["Title"] = df["Title"].replace({
        "Mlle": "Miss",
        "Ms": "Miss",
        "Mme": "Mrs"
    })

    common_titles = ["Mr", "Miss", "Mrs", "Master"]
    df["Title"] = df["Title"].where(df["Title"].isin(common_titles), "Rare")

    df["CabinDeck"] = df["Cabin"].str[0]
    df["HasCabin"] = df["Cabin"].notna().astype(int)

    df["TicketGroupSize"] = df.groupby("Ticket")["Ticket"].transform("count")

    df["FarePerPerson"] = df["Fare"] / df["TicketGroupSize"].replace(0, 1)

    df["Surname"] = df["Name"].str.split(",").str[0].str.strip()

    df["AgeMissing"] = df["Age"].isna().astype(int)
    df["FareMissing"] = df["Fare"].isna().astype(int)
    df["EmbarkedMissing"] = df["Embarked"].isna().astype(int)

    df["Child"] = ((df["Age"] < 16) & df["Age"].notna()).astype(int)

    df["Mother"] = (
        (df["Sex"] == "female") &
        (df["Parch"] > 0) &
        (df["Age"] > 18) &
        (df["Age"] < 50) &
        (df["Title"] != "Miss")
    ).astype(int)

    df["Sex_Pclass"] = df["Sex"].astype(str) + "_" + df["Pclass"].astype(str)

    df["FamilySizeBand"] = pd.cut(
        df["FamilySize"],
        bins=[0, 1, 4, 7, 20],
        labels=["Alone", "Small", "Medium", "Large"]
    ).astype(str)

    df["AgeBand"] = pd.cut(
        df["Age"],
        bins=[0, 12, 18, 30, 50, 100],
        labels=["Child", "Teen", "YoungAdult", "Adult", "Senior"]
    ).astype(str)

    df["FareBand"] = pd.qcut(
        df["Fare"],
        q=5,
        labels=False,
        duplicates="drop"
    )

    return df


In [3]:
train_fe = add_features(train)

train_fe.shape


(891, 29)

In [4]:
target = "Survived"

features = [
    "Pclass",
    "Sex",
    "Age",
    "SibSp",
    "Parch",
    "Fare",
    "Embarked",
    "FamilySize",
    "IsAlone",
    "Title",
    "CabinDeck",
    "HasCabin",
    "TicketGroupSize",
    "FarePerPerson",
    "Surname",
    "AgeMissing",
    "FareMissing",
    "EmbarkedMissing",
    "Child",
    "Mother",
    "Sex_Pclass",
    "FamilySizeBand",
    "AgeBand",
    "FareBand"
]

X = train_fe[features].copy()
y = train_fe[target]

categorical_columns = X.select_dtypes(include=["object", "category"]).columns

for column in categorical_columns:
    X[column] = X[column].astype("category").cat.codes

X = X.replace(-1, np.nan)

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training shape:", X_train.shape)
print("Validation shape:", X_valid.shape)


C:\Users\aakif\AppData\Local\Temp\ipykernel_12112\3660854951.py:33: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(include=["object", "category"]).columns


Training shape: (712, 24)
Validation shape: (179, 24)


In [5]:
model = XGBClassifier(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.03,
    subsample=0.85,
    colsample_bytree=0.85,
    min_child_weight=2,
    gamma=0.1,
    reg_alpha=0.05,
    reg_lambda=1.5,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

predictions = model.predict(X_valid)

accuracy = accuracy_score(y_valid, predictions)

print(f"Validation accuracy: {accuracy:.4f}")


Validation accuracy: 0.7765


In [6]:
feature_importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

feature_importance.head(20)


Sex_Pclass         0.221944
Sex                0.220312
Pclass             0.092894
Child              0.055143
CabinDeck          0.043509
HasCabin           0.043471
FamilySize         0.034055
Title              0.031421
TicketGroupSize    0.028405
SibSp              0.028051
Age                0.026109
Embarked           0.025890
FarePerPerson      0.023008
AgeBand            0.021683
Surname            0.020445
FamilySizeBand     0.020061
Fare               0.019501
Parch              0.013789
FareBand           0.010646
Mother             0.010582
dtype: float32

## Result

Compare this experiment against:

- Experiment 01: 0.8045
- Experiment 02: 0.8630

The goal is to keep improving the model before creating any submission.
